- 记忆
- 对话检索链
- 定义一个适用于个人文档的聊天机器人

In [1]:
### 加载已持久化的数据库
from langchain_chroma.vectorstores import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

persist_directory = 'data/matplotlib/'
embedding = HuggingFaceEmbeddings()
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

question = "这门课的主要内容是什么？"
docs = vectordb.similarity_search(question,k=3)
print(len(docs))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3


In [2]:
### 创建一个llm
from langchain_ollama import ChatOllama
from langchain_classic.chains import RetrievalQA

llm = ChatOllama(model='qwen2.5:7b', temperature=0.0)
llm.invoke("你好")

AIMessage(content='你好！有什么问题或者需要帮助的吗？', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-03-30T09:22:50.4821196Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8442307900, 'load_duration': 8149256400, 'prompt_eval_count': 30, 'prompt_eval_duration': 70801400, 'eval_count': 11, 'eval_duration': 190392900, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--019d3e0d-72dd-70a2-9be8-7cf2862a917a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 11, 'total_tokens': 41})

In [3]:
### 构建prompt template
from langchain_core.prompts import PromptTemplate

template = """
使用以下上下文来回答最后的问题。
如果你不知道答案，就说你不知道，不要试图编造答案。
最多使用三句话。尽量使答案简明扼要。
总是在回答的最后说“谢谢你的提问！”。

上下文：```{context}```

问题: ```{question}```

有用的回答:
"""

QA_CHAIN_PROMPT = PromptTemplate(input_variables=["context", "question"],template=template,)

### 运行chain
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

question = "这门课的主题是什么？"
result = qa_chain.invoke(question)
print(result["result"])

这门课的主题是关于matplotlib的使用，包括绘图样式和颜色的设置。谢谢你的提问！


Memory 记忆

In [5]:
### ConversationBufferMemory: 保存聊天消息历史的列表，将在回答时和问题一起输入并添加到上下文中

from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history", # 与 prompt 的输入变量保持一致。
    return_messages=True # 将以消息列表的形式返回聊天记录，而不是单个字符串
)

C:\Users\61075\AppData\Local\Temp\ipykernel_32264\2971817537.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


ConversationalRetrievalChain 对话检索链

In [7]:
### 将历史对话与当前新问题合并成一个完整的query语句
### 在db中搜索该query的相关文档
### 获取结果后，存储所有答案到对话记忆区
### 可以查看完整的对话流程

from langchain_classic.chains import ConversationalRetrievalChain

retrieval = vectordb.as_retriever()
qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retrieval,
    memory=memory,
)

question = "这门课会学习 python 吗？"
result = qa.invoke(question)
print(result)

{'question': '这门课会学习 python 吗？', 'chat_history': [HumanMessage(content='这门课会学习 python 吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='这门课程主要围绕 matplotlib 进行讲解，包括 Figure 和 Axes 的组成、绘图接口以及样式和颜色的使用等。虽然会用到 Python 来编写代码示例，但主要目的是为了展示如何在 matplotlib 中进行数据可视化，并不是专门教授 Python 语言本身。所以严格来说，这门课不会系统地学习 Python 编程语言。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'answer': '这门课程主要围绕 matplotlib 进行讲解，包括 Figure 和 Axes 的组成、绘图接口以及样式和颜色的使用等。虽然会用到 Python 来编写代码示例，但主要目的是为了展示如何在 matplotlib 中进行数据可视化，并不是专门教授 Python 语言本身。所以严格来说，这门课不会系统地学习 Python 编程语言。'}


In [9]:
result["answer"]

'这门课程主要围绕 matplotlib 进行讲解，包括 Figure 和 Axes 的组成、绘图接口以及样式和颜色的使用等。虽然会用到 Python 来编写代码示例，但主要目的是为了展示如何在 matplotlib 中进行数据可视化，并不是专门教授 Python 语言本身。所以严格来说，这门课不会系统地学习 Python 编程语言。'

In [10]:
question = "为什么这门课需要这个前提？"
result = qa.invoke(question)
print(result['answer'])

根据提供的上下文信息，没有直接提到这门课程为何需要Python前提条件。所提供的内容主要围绕matplotlib库的功能和使用方法展开，并未提及课程对编程语言的具体要求。因此，基于现有信息，我无法回答“这门课为什么需要Python的前提条件”这个问题。
